# Döntési fa regresszor — Seoul Bike Trip Duration

**Felelős:** 3. hallgató

Ez a notebook egy DecisionTreeRegressor-t tanít be és hangol GridSearchCV-vel, majd menti az eredményt a közös `metrics.csv`-be.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/03_decision_tree.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git pull
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
import joblib

from src.config import RANDOM_SEED, CV_FOLDS, MODELS_DIR, FIGURES_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.evaluation import append_metrics, evaluate_model
from src.visualization import (
    plot_feature_importance,
    plot_predicted_vs_actual,
    plot_residuals,
)

ensure_dirs()

## 2. Adatbetöltés

A v1 final pipeline kimenetét töltjük be `load_v1_data()`-vel.

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_v1_data()

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Target — train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, '
      f'min={y_train.min()}, max={y_train.max()}')

## 3. Baseline: alap döntési fa

Default `DecisionTreeRegressor` paraméterhangolás nélkül — referenciapont a tuned változathoz.

In [ ]:
dt_default = DecisionTreeRegressor(random_state=RANDOM_SEED)

result_default, y_pred_default = evaluate_model(
    dt_default,
    X_train, y_train, X_test, y_test,
    model_name='Decision Tree (default)',
    notes='no tuning',
)

print(f'MAE: {result_default.mae:.3f}')
print(f'RMSE: {result_default.rmse:.3f}')
print(f'R²: {result_default.r2:.3f}')
print(f'Train idő: {result_default.train_time_sec:.2f} s')

append_metrics(result_default)

## 4. GridSearchCV hangolás

5×3×3 = 45 kombináció × 5 fold cross-validation, neg-MAE szerint.

In [ ]:
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 10, 50],
    'min_samples_leaf': [1, 10, 50],
}

dt = DecisionTreeRegressor(random_state=RANDOM_SEED)

grid = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
)

result_tuned, y_pred_tuned = evaluate_model(
    grid,
    X_train, y_train, X_test, y_test,
    model_name='Decision Tree (tuned)',
    notes=f'GridSearchCV cv={CV_FOLDS}',
)

print(f'\nLegjobb paraméterek: {grid.best_params_}')
print(f'CV legjobb score (neg MAE): {grid.best_score_:.3f}')
print(f'\nTeszt MAE: {result_tuned.mae:.3f}')
print(f'Teszt RMSE: {result_tuned.rmse:.3f}')
print(f'Teszt R²: {result_tuned.r2:.3f}')

append_metrics(result_tuned)

best_model = grid.best_estimator_
joblib.dump(best_model, MODELS_DIR / 'decision_tree_tuned.joblib')

## 5. Vizualizációk

In [ ]:
plot_predicted_vs_actual(
    y_test, y_pred_tuned,
    model_name='Decision Tree (tuned)',
    save_path='dt_predicted_vs_actual.png',
)
plt.show()

In [ ]:
plot_residuals(
    y_test, y_pred_tuned,
    model_name='Decision Tree (tuned)',
    save_path='dt_residuals.png',
)
plt.show()

In [ ]:
plot_feature_importance(
    feature_names=feature_names,
    importances=best_model.feature_importances_,
    model_name='Decision Tree (tuned)',
    top_n=20,
    save_path='dt_feature_importance.png',
)
plt.show()

## 6. Tanulságok

_Hát az van_